In [7]:
from Models.modl import modl
from torch.utils.data import DataLoader
from Transforms import (pad, trim_coils, combine_coil, toTensor, permute, 
                        view_as_real, remove_slice_dim, fft_2d, normalize, addChannels)
from Dataset.undersampled_dataset import UndersampledKSpaceDataset
from torchvision.transforms import Compose
import numpy as np
import torch
from Models.varnet import VarNet

In [ ]:
%load_ext autoreload
%autoreload 2 

In [4]:
def collate_fn(data):
    undersampled = [d['undersampled'] for d in data]
    sampled = [d['k_space'] for d in data]
    ismrmrd_header = [d['ismrmrd_header'] for d in data]
    mask = [d['mask'] for d in data]
    recon_rss = [d['reconstruction_rss'] for d in data]

    undersampled = torch.concat(undersampled, dim=0)
    sampled = torch.concat(sampled, dim=0)
    mask = torch.concat(mask, dim=0)

    data = {
        'undersampled': undersampled, 
        'sampled': sampled,
        'ismrmrd_header': ismrmrd_header,
        'mask': mask, 
        'recon': recon_rss,
    }
    return data

In [6]:
transforms = Compose(
    (
        trim_coils(12),
        pad((640, 320)), 
        fft_2d(axes=[2,3]),
        combine_coil(),
        normalize(),
        toTensor(),
    )
)
dataset = UndersampledKSpaceDataset('/home/kadotab/projects/def-mchiew/kadotab/dataset/multicoil_train', transforms=transforms)
dataloader = DataLoader(dataset, batch_size=1, collate_fn=collate_fn)
    

In [8]:
device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')


In [5]:
data = (next(iter(dataloader)))

In [6]:
import cProfile
cProfile.run('next(iter(dataloader))', 'dataloader.profile')

In [10]:
model = VarNet(2, 2)
model.to(device)

VarNet(
  (cascade): ModuleList(
    (0): VarnetBlock(
      (unet): Unet(
        (down_sample_layers): ModuleList(
          (0): double_conv(
            (conv1): Conv2d(2, 18, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
            (conv2): Conv2d(18, 18, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
            (activation): ReLU()
            (instance_norm1): InstanceNorm2d(18, eps=1e-05, momentum=0.1, affine=False, track_running_stats=False)
            (instance_norm2): InstanceNorm2d(18, eps=1e-05, momentum=0.1, affine=False, track_running_stats=False)
            (drop_out1): Dropout2d(p=0, inplace=False)
            (drop_out2): Dropout2d(p=0, inplace=False)
          )
          (1): Unet_down(
            (down): down(
              (max_pool): MaxPool2d(kernel_size=2, stride=(2, 2), padding=0, dilation=1, ceil_mode=False)
            )
            (conv): double_conv(
              (conv1): Conv2d(18, 36, kernel_size=(3, 3), stride=(1,

In [14]:
print(sum(param.numel() for param in model.parameters()))

15212653


In [12]:
loss_fn = torch.nn.MSELoss()
optimizer = torch.optim.SGD(model.parameters(), momentum=0.99, lr=0.0001)

In [15]:
def train(model, loss_function, optimizer, dataloader):
    cur_loss = 0
    current_index = 0
    for data in dataloader:
        
        sampled = data['sampled']
        mask = data['mask']
        undersampled = data['undersampled']
        for i in range(sampled.shape[0]):
            optimizer.zero_grad()
            sampled_slice = sampled[[i],...]
            mask_slice = mask[[i],...]
            undersampled_slice = undersampled[[i],...]
            mask_slice = mask_slice.to(device)
            undersampled_slice = undersampled_slice.to(device)

            predicted_sampled = model(undersampled_slice, mask_slice)
            loss = loss_function(torch.view_as_real(predicted_sampled), torch.view_as_real(sampled_slice))

            loss.backward()
            optimizer.step()
            cur_loss += loss.item()
            if current_index % 10 == 9:
                print(f"Iteration: {current_index + 1:>d}, Loss: {cur_loss:>7f}")
                cur_loss = 0
            current_index += 1


In [16]:
train(model, loss_fn, optimizer, dataloader)

/home/kadotab/python/mri_machine_learning_reconstruction/Transforms/transforms.py:56: RuntimeWarning: invalid value encountered in divide
  coil_sense = temp/full_img
/home/kadotab/python/mri_machine_learning_reconstruction/Transforms/transforms.py:91: RuntimeWarning: invalid value encountered in divide
  return sample/sample.max(axis=(-1, -2)).reshape((-1, 1, 1))


IndexError: tuple index out of range